# 02. Appropriations Intent: Node Classification

Classify BillNode text by appropriations intent (appropriation, restriction, transfer, rescission, cap).
Then build an enriched financial summary that adds labels to the diff's dollar-amount changes.


In [4]:
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")

from bill_tree import bill_title, normalize_bill

## Load sample bill
H.R. 4366 - 118th Congress

In [5]:
# Load bill
xml_path = Path("../bills/118-hr-4366/1_reported-in-house.xml")
tree = normalize_bill(xml_path)

## Build rule-based classifier

In [ ]:
# Node-opening patterns
RESTRICT = re.compile(r"^\s*None of the funds", re.IGNORECASE)
TRANSFER = re.compile(r"^\s*Of (?:the )?amounts", re.IGNORECASE)
APPROP = re.compile(r"^\s*For\b", re.IGNORECASE | re.DOTALL)
RESCISSION = re.compile(r"is hereby rescinded", re.IGNORECASE)
DIRECTIVE = re.compile(r"^\s*The\s+\w[\w\s]+(?:shall|may not)\b", re.IGNORECASE)
REPROGRAM = re.compile(r"^\s*no project may be (?:increased|decreased)", re.IGNORECASE)
DELAYED_APPROP = re.compile(r"^\s*\$[\d,]+.{0,50}\bshall become available\b", re.IGNORECASE | re.DOTALL)

# Sub-clause patterns
PROVIDED_RE = re.compile(r"\bProvided(?:\s+further)?,?\s+That\b", re.IGNORECASE)
EARMARK = re.compile(r"of the amount.{0,50}under this heading.{0,100}specified in the table", re.IGNORECASE | re.DOTALL)
AVAILABILITY = re.compile(r"of the amount.{0,100}shall remain available until", re.IGNORECASE | re.DOTALL)
SUB_ALLOC = re.compile(r"^\s*,?\s*\$[\d,]+\s+shall\s+be\s+(?:for|available)", re.IGNORECASE)
CAP = re.compile(r"not (?:more than|to exceed)\s+\$[\d,]+", re.IGNORECASE)
OF_WHICH_AVAIL = re.compile(r"^\s*of which.{0,80}\bshall remain available\b", re.IGNORECASE | re.DOTALL)
OF_WHICH_ALLOC = re.compile(r"^\s*of which\b", re.IGNORECASE)

# Splitting / extraction helpers
OF_WHICH_RE = re.compile(r",?\s*\bof which\b", re.IGNORECASE)
IN_ADDITION_RE = re.compile(r";\s*and,?\s*in addition,", re.IGNORECASE)
CAP_AMOUNT_RE = re.compile(r"not (?:more than|to exceed)\s+\$[\d,]+(?:\.\d+)?", re.IGNORECASE)
DOLLAR = re.compile(r"\$([\d,]+(?:\.\d+)?)")


def classify_text(text):
    if not text:
        return None
    if RESTRICT.match(text):
        return "restriction"
    if TRANSFER.match(text):
        return "transfer"
    if APPROP.match(text):
        return "rescission" if RESCISSION.search(text) else "appropriation"
    if RESCISSION.search(text):
        return "rescission"
    if DIRECTIVE.match(text):
        return "directive"
    if REPROGRAM.match(text):
        return "cap"
    if DELAYED_APPROP.match(text):
        return "appropriation"
    if EARMARK.search(text):
        return "earmark"
    if AVAILABILITY.search(text):
        return "availability"
    if SUB_ALLOC.match(text):
        return "sub_allocation"
    if OF_WHICH_AVAIL.match(text):
        return "availability"
    if OF_WHICH_ALLOC.match(text):
        return "sub_allocation"
    if CAP.search(text):
        return "cap"
    return "unknown"


def primary_amount(text):
    if not text:
        return None
    pre_provided = text.split("Provided")[0]
    m = DOLLAR.search(pre_provided)
    return float(m.group(1).replace(",", "")) if m else None


def split_clauses(text):
    """Split on Provided That → ; and in addition → of which, in that order."""
    if not text:
        return []
    results = []
    for i, provided_part in enumerate(PROVIDED_RE.split(text)):
        level = "primary" if i == 0 else "sub"
        for j, addition_part in enumerate(IN_ADDITION_RE.split(provided_part)):
            of_which_parts = OF_WHICH_RE.split(addition_part)
            for k, clause in enumerate(of_which_parts):
                sub_level = level if (j == 0 and k == 0) else "sub"
                prefix = "of which " if k > 0 else ""
                results.append((prefix + clause.strip(), sub_level))
    return results


def non_cap_amounts(text):
    """Dollar amounts not preceded by cap language."""
    return DOLLAR.findall(CAP_AMOUNT_RE.sub("", text))

### Node-level classification

Classify every dollar-amount node of test bill.
Review `unknown` rows to find gaps in the patterns.

In [7]:
dollar_nodes = [n for n in tree.nodes if DOLLAR.search(n.body_text or "")]

rows = [
    {
        "label": classify_text(n.body_text),
        "amount": primary_amount(n.body_text),
        "path": " > ".join(n.display_path[-2:]) if n.display_path else "",
        "preview": (n.body_text or "")[:150],
    }
    for n in dollar_nodes
]

df_nodes = pd.DataFrame(rows)
print(df_nodes["label"].value_counts())
df_nodes

label
appropriation    51
restriction       5
directive         4
transfer          4
cap               2
unknown           1
Name: count, dtype: int64


,label,amount,path,preview
0,appropriation,1.517455e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
1,appropriation,4.477961e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
2,appropriation,2.439614e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
3,appropriation,2.651047e+09,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For acquisition, construction, installation, a..."
4,appropriation,3.692610e+08,TITLE I—DEPARTMENT OF DEFENSE > Military const...,"For construction, acquisition, expansion, reha..."
...,...,...,...,...
62,appropriation,4.720000e+07,United states court of appeals for veterans cl...,For necessary expenses for the operation of th...
63,appropriation,2.000000e+03,"Cemeterial expenses, army > SALARIES AND EXPENSES","For necessary expenses for maintenance, operat..."
64,appropriation,8.860000e+07,"Cemeterial expenses, army > CONSTRUCTION",For necessary expenses for planning and design...
65,appropriation,7.700000e+07,Armed forces retirement home > TRUST FUND,For expenses necessary for the Armed Forces Re...


In [8]:
# Review unknowns
df_nodes[df_nodes["label"] == "unknown"][["path", "preview"]]

,path,preview
66,GENERAL PROVISIONS > sec. 418,$0.


In [9]:
# Investigate 'unknown' node
node = dollar_nodes[66]
print("display_path:", node.display_path)
print("header_text: ", node.header_text)
print("section_number:", node.section_number)
print("body_text:", node.body_text)

display_path: ('TITLE IV', 'GENERAL PROVISIONS', 'sec. 418')
header_text:  
section_number: Sec. 418
body_text: $0.


The text for the unknown node is just "$0". Place holder?

## Multi-amount classification
Expand classifier capabilities to identify the "primary" and "sub" (modifications to the primary amount, like caps or earmarks) dollar amounts in nodes containing more than one $ value.

In [10]:
multi_dollar_nodes = [n for n in tree.nodes if len(DOLLAR.findall(n.body_text or "")) > 1]

print(f"{len(multi_dollar_nodes)} nodes with multiple dollar amounts")

for n in multi_dollar_nodes:
    clauses = re.split(r"\bProvided(?:\s+further)?,?\s+That\b", n.body_text or "", flags=re.IGNORECASE)
    print(f"\n{' > '.join(n.display_path[-2:])}")
    for i, clause in enumerate(clauses):
        amounts = DOLLAR.findall(clause)
        if amounts:
            label = "primary" if i == 0 else classify_text(clause)
            print(f"  [{label}] ${amounts} — {clause.strip()}")

28 nodes with multiple dollar amounts

TITLE I—DEPARTMENT OF DEFENSE > Military construction, army
  [primary] $['1,517,455,000,'] — For acquisition, construction, installation, and equipment of temporary or permanent public works, military installations, facilities, and real property for the Army as currently authorized by law, including personnel in the Army Corps of Engineers and other personal services necessary for the purposes of this appropriation, and for construction and operation of facilities in support of the functions of the Commander in Chief, $1,517,455,000, to remain available until September 30, 2028:
  [cap] $['345,775,000'] — , of this amount, not to exceed $345,775,000 shall be available for study, planning, design, architect and engineer services, and host nation support, as authorized by law, unless the Secretary of the Army determines that additional obligations are necessary for such purposes and notifies the Committees on Appropriations of both Houses of Congre

In [11]:
PRIMARY_LABELS = {"appropriation", "transfer", "rescission"}
primary_nodes = [n for n in tree.nodes if classify_text(n.body_text) in PRIMARY_LABELS]

rows = []
for n in primary_nodes:
    account = " > ".join(n.display_path[-2:]) if n.display_path else ""
    for clause_text, level in split_clauses(n.body_text):
        if not DOLLAR.search(clause_text):
            continue
        clean = non_cap_amounts(clause_text)
        m = DOLLAR.search(CAP_AMOUNT_RE.sub("", clause_text)) or DOLLAR.search(clause_text)
        amount = float(m.group(1).replace(",", "")) if m else None
        rows.append(
            {
                "account": account,
                "level": level,
                "type": classify_text(n.body_text) if level == "primary" else classify_text(clause_text),
                "amount": amount,
                "needs_review": len(clean) > 1,
                "preview": clause_text[:150],
                "body_text": n.body_text or "",
            }
        )

df_financial = pd.DataFrame(rows)

df_financial.head(15)

,account,level,type,amount,needs_review,preview,body_text
0,TITLE I—DEPARTMENT OF DEFENSE > Military const...,primary,appropriation,1.517455e+09,False,"For acquisition, construction, installation, a...","For acquisition, construction, installation, a..."
1,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,cap,3.457750e+08,False,", of this amount, not to exceed $345,775,000 s...","For acquisition, construction, installation, a..."
2,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,earmark,1.629000e+08,False,of the amount made available under this headin...,"For acquisition, construction, installation, a..."
3,TITLE I—DEPARTMENT OF DEFENSE > Military const...,primary,appropriation,4.477961e+09,False,"For acquisition, construction, installation, a...","For acquisition, construction, installation, a..."
4,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,cap,6.026250e+08,False,", of this amount, not to exceed $602,625,000 s...","For acquisition, construction, installation, a..."
5,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,earmark,5.268300e+07,False,of the amount made available under this headin...,"For acquisition, construction, installation, a..."
6,TITLE I—DEPARTMENT OF DEFENSE > Military const...,primary,appropriation,2.439614e+09,False,"For acquisition, construction, installation, a...","For acquisition, construction, installation, a..."
7,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,cap,4.506140e+08,False,", of this amount, not to exceed $450,614,000 s...","For acquisition, construction, installation, a..."
8,TITLE I—DEPARTMENT OF DEFENSE > Military const...,sub,earmark,3.030000e+07,False,of the amount made available under this headin...,"For acquisition, construction, installation, a..."
9,TITLE I—DEPARTMENT OF DEFENSE > Military const...,primary,appropriation,2.651047e+09,False,"For acquisition, construction, installation, a...","For acquisition, construction, installation, a..."


In [12]:
df_financial.loc[df_financial["needs_review"] == 1]

,account,level,type,amount,needs_review,preview,body_text
36,Administrative provisions > sec. 130,primary,appropriation,25000000.0,True,For an additional amount for the accounts and ...,For an additional amount for the accounts and ...
37,Administrative provisions > sec. 131,primary,appropriation,65000000.0,True,For an additional amount for the accounts and ...,For an additional amount for the accounts and ...


In [13]:
# Check for unknowns after separating clauses
unknowns = df_financial.loc[df_financial["type"] == "unknown", ["account", "level", "type", "amount", "preview"]]

for row in unknowns.itertuples(index=False):
    print(f"{row.preview}")

In [14]:
df_financial.columns

Index(['account', 'level', 'type', 'amount', 'needs_review', 'preview',
       'body_text'],
      dtype='str')

## Generate sample report (HTML)

In [ ]:
from datetime import datetime
from html import escape as esc

TYPE_COLORS = {
    "appropriation": ("#dcfce7", "#166534"),
    "transfer": ("#cffafe", "#155e75"),
    "rescission": ("#fee2e2", "#991b1b"),
    "restriction": ("#fee2e2", "#991b1b"),
    "cap": ("#fef3c7", "#92400e"),
    "earmark": ("#ede9fe", "#5b21b6"),
    "availability": ("#d1fae5", "#065f46"),
    "sub_allocation": ("#dbeafe", "#1e40af"),
    "directive": ("#e0e7ff", "#3730a3"),
    "unknown": ("#f3f4f6", "#374151"),
}

TYPE_DESCRIPTIONS = {
    "appropriation": "Funds directly allocated to an agency or program",
    "transfer": "Funds moved between accounts",
    "rescission": "Previously appropriated funds clawed back",
    "restriction": "Prohibition on how funds may be used",
    "cap": "Upper limit on spending for a specific purpose",
    "earmark": "Funds designated for a recipient listed in a table",
    "availability": "Time extension specifying when funds may be spent",
    "sub_allocation": "Portion of funds reserved for a specific use",
    "directive": "Mandatory action required of an agency",
    "unknown": "Could not be automatically classified",
}

PROVISION_TYPES = ["appropriation", "transfer", "rescission"]
SUB_CLAUSE_TYPES = ["cap", "earmark", "availability", "sub_allocation", "restriction", "directive", "unknown"]


def fmt_amount(x):
    return f"${x:,.0f}" if pd.notna(x) else "—"


def type_badge(t):
    bg, fg = TYPE_COLORS.get(t, ("#f3f4f6", "#374151"))
    return (
        f'<span style="background:{bg};color:{fg};padding:2px 7px;'
        f'border-radius:999px;font-size:11px;font-weight:600">{esc(t)}</span>'
    )


def highlight_body(body_text):
    pieces = re.split(r"(\bProvided(?:\s+further)?,?\s+That\b)", body_text, flags=re.IGNORECASE)
    clause_idx = 0
    out = []
    for piece in pieces:
        if re.match(r"\bProvided(?:\s+further)?,?\s+That\b", piece, re.IGNORECASE):
            out.append(f'<strong style="color:#555">{esc(piece)}</strong> ')
        else:
            t = classify_text(body_text) if clause_idx == 0 else classify_text(piece)
            t = t or "unknown"
            bg, fg = TYPE_COLORS.get(t, ("#f3f4f6", "#374151"))
            out.append(
                f'<span style="background:{bg};color:{fg};border-radius:3px;padding:1px 3px">{esc(piece)}</span>'
            )
            clause_idx += 1
    return "".join(out)


def legend_section(title, types):
    items = "".join(
        f'<div style="display:flex;align-items:baseline;gap:10px;padding:3px 0">'
        f"{type_badge(t)}"
        f'<span style="color:#555;font-size:13px">{TYPE_DESCRIPTIONS[t]}</span></div>'
        for t in types
    )
    return (
        f"<div>"
        f'<div style="font-size:11px;text-transform:uppercase;letter-spacing:0.06em;'
        f'color:#888;font-weight:600;margin-bottom:6px">{title}</div>'
        f"{items}</div>"
    )


def build_legend():
    return (
        '<div style="margin-bottom:20px;border:1px solid #e3ddd7;border-radius:0.625rem;'
        'padding:12px 14px;background:#fff">'
        '<div style="font-weight:600;font-size:14px;margin-bottom:10px">Category key</div>'
        '<div style="display:grid;grid-template-columns:1fr 1fr;gap:16px">'
        + legend_section("Provision types", PROVISION_TYPES)
        + legend_section("Sub-clause types", SUB_CLAUSE_TYPES)
        + "</div></div>"
    )


def ordinal(n):
    if 11 <= n % 100 <= 13:
        return f"{n}th"
    return f"{n}{['th', 'st', 'nd', 'rd', 'th'][min(n % 10, 4)]}"


def build_financial_html(df, tree):
    FONT_SANS = "ui-sans-serif, system-ui, -apple-system, 'Segoe UI', Roboto, Arial, sans-serif"
    FONT_MONO = "ui-monospace, 'SF Mono', Menlo, Consolas, monospace"

    css = f"""<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ background: #f9f7f5; font-family: {FONT_SANS}; color: #1c1c3a;
       font-size: 14px; line-height: 1.6; margin: 24px; }}
h2 {{ font-family: ui-serif, Georgia, serif; font-size: 20px; margin-bottom: 12px;
     letter-spacing: -0.02em; }}
table {{ width: 100%; border-collapse: collapse; background: #fff;
         border-radius: 0.625rem; table-layout: fixed;
         box-shadow: 0 1px 2px rgba(28,28,58,.04), 0 1px 3px rgba(28,28,58,.06);
         overflow: hidden; }}
th {{ background: #eeeae6; text-align: left; padding: 9px 12px;
      border-bottom: 2px solid #e3ddd7; font-weight: 600; font-size: 13px; }}
td {{ padding: 8px 12px; border-bottom: 1px solid #e3ddd7; vertical-align: middle; }}
.amount {{ text-align: right; font-family: {FONT_MONO};
           font-variant-numeric: tabular-nums; white-space: nowrap; }}
.row-primary {{ background: #f0ede9; font-weight: 600; cursor: pointer; user-select: none; }}
.row-primary:hover {{ filter: brightness(0.97); }}
.row-primary td:first-child::before {{ content: '▶'; font-size: 10px; margin-right: 8px;
                                       color: #888; display: inline-block;
                                       transition: transform 0.15s; }}
.row-primary.open td:first-child::before {{ transform: rotate(90deg); }}
.detail-inner {{ display: flex; }}
.detail-amounts {{ min-width: 220px; max-width: 260px; padding: 12px;
                   border-right: 1px solid #e3ddd7; display: flex;
                   flex-direction: column; gap: 10px; }}
.detail-amount-item {{ display: flex; flex-direction: column; gap: 3px; }}
.detail-amount-value {{ font-family: {FONT_MONO}; font-size: 15px;
                        font-variant-numeric: tabular-nums; }}
.detail-text {{ padding: 12px 16px; font-size: 13px; line-height: 1.9;
                white-space: pre-wrap; flex: 1; }}
.badge-review {{ background: #fef3c7; color: #92400e; font-size: 11px;
                 padding: 1px 6px; border-radius: 999px; margin-left: 6px; font-weight: 500; }}
.count-badge {{ font-size: 11px; color: #888; font-weight: 400; margin-left: 6px; }}
.sort-bar {{ display: flex; gap: 8px; align-items: center; margin-bottom: 12px; }}
.sort-bar span {{ font-size: 13px; color: #888; margin-right: 2px; }}
.sort-btn {{ padding: 4px 12px; border: 1px solid #e3ddd7; border-radius: 999px;
             background: #fff; font-size: 13px; cursor: pointer;
             font-family: inherit; color: #1c1c3a; }}
.sort-btn:hover {{ background: #eeeae6; }}
.sort-btn.active {{ background: #1c1c3a; color: #fff; border-color: #1c1c3a; }}
</style>"""

    js = """<script>
function toggleDetail(el) {
    var id = el.dataset.group;
    var open = el.classList.toggle('open');
    document.getElementById('detail-' + id).style.display = open ? '' : 'none';
    var state = JSON.parse(localStorage.getItem('fin-open') || '{}');
    state[id] = open;
    localStorage.setItem('fin-open', JSON.stringify(state));
}

function sortTable(key, btn) {
    var tbody = document.querySelector('tbody');
    var rows = Array.from(tbody.querySelectorAll('tr'));
    var groups = [];
    var i = 0;
    while (i < rows.length) {
        if (rows[i].classList.contains('row-primary')) {
            groups.push([rows[i], rows[i + 1]]);
            i += 2;
        } else { i++; }
    }
    if (key === 'amount') {
        groups.sort(function(a, b) {
            return parseFloat(b[0].dataset.amount || 0) - parseFloat(a[0].dataset.amount || 0);
        });
    } else if (key === 'review') {
        groups.sort(function(a, b) {
            return (b[0].dataset.review === '1' ? 1 : 0) - (a[0].dataset.review === '1' ? 1 : 0);
        });
    } else {
        groups.sort(function(a, b) {
            return parseInt(a[0].dataset.group) - parseInt(b[0].dataset.group);
        });
    }
    var frag = document.createDocumentFragment();
    groups.forEach(function(g) { frag.appendChild(g[0]); frag.appendChild(g[1]); });
    tbody.appendChild(frag);
    document.querySelectorAll('.sort-btn').forEach(function(b) { b.classList.remove('active'); });
    btn.classList.add('active');
}

window.addEventListener('DOMContentLoaded', function() {
    var state = JSON.parse(localStorage.getItem('fin-open') || '{}');
    Object.keys(state).forEach(function(id) {
        if (state[id]) {
            var row = document.querySelector('[data-group="' + id + '"]');
            if (row) {
                row.classList.add('open');
                document.getElementById('detail-' + id).style.display = '';
            }
        }
    });
});
</script>"""

    groups = []
    current = []
    for _, row in df.iterrows():
        if row["level"] == "primary":
            if current:
                groups.append(current)
            current = [row]
        else:
            current.append(row)
    if current:
        groups.append(current)

    rows_html = []
    for gid, group in enumerate(groups):
        primary = group[0]
        subs = group[1:]
        needs_review = any(r["needs_review"] for r in group)
        review_badge = '<span class="badge-review">⚠ review</span>' if needs_review else ""
        count_badge = f'<span class="count-badge">({len(subs)})</span>' if subs else ""
        amount_val = primary["amount"] if pd.notna(primary["amount"]) else 0

        rows_html.append(
            f'<tr class="row-primary" onclick="toggleDetail(this)" data-group="{gid}"'
            f' data-amount="{amount_val}" data-review="{"1" if needs_review else "0"}">'
            f"<td>{esc(primary['account'])}{count_badge}{review_badge}</td>"
            f"<td>{type_badge(primary['type'])}</td>"
            f'<td class="amount">{fmt_amount(primary["amount"])}</td>'
            f"</tr>"
        )

        amount_items = "".join(
            f'<div class="detail-amount-item">'
            f"{type_badge(r['type'])}"
            f'<span class="detail-amount-value">{fmt_amount(r["amount"])}</span>'
            f"</div>"
            for r in group
        )
        highlighted = highlight_body(primary["body_text"]) if primary["body_text"] else ""

        rows_html.append(
            f'<tr id="detail-{gid}" style="display:none">'
            f'<td colspan="3" style="padding:0">'
            f'<div class="detail-inner">'
            f'<div class="detail-amounts">{amount_items}</div>'
            f'<div class="detail-text">{highlighted}</div>'
            f"</div></td></tr>"
        )

    sort_bar = (
        '<div class="sort-bar"><span>Sort:</span>'
        '<button class="sort-btn active" onclick="sortTable(\'default\', this)">Default</button>'
        '<button class="sort-btn" onclick="sortTable(\'amount\', this)">Amount ↓</button>'
        '<button class="sort-btn" onclick="sortTable(\'review\', this)">Needs review first</button>'
        "</div>"
    )
    bill_header = (
        f'<div style="margin-bottom:20px">'
        f'<h1 style="font-family:ui-serif,Georgia,serif;font-size:24px;'
        f'letter-spacing:-0.02em;margin-bottom:4px">{esc(bill_title(tree))}</h1>'
        f'<p style="color:#888;font-size:14px">{ordinal(tree.congress)} Congress</p>'
        f"</div>"
    )
    return (
        css
        + js
        + bill_header
        + build_legend()
        + "<h2>Financial Summary</h2>"
        + sort_bar
        + "<table><thead><tr>"
        + '<th style="width:75%">Account</th>'
        + '<th style="width:12%">Type</th>'
        + '<th style="width:13%">Amount</th>'
        + "</tr></thead>"
        + "<tbody>"
        + "".join(rows_html)
        + "</tbody></table>"
    )


ts = datetime.now().strftime("%H:%M:%S")
html = f'<p style="color:#999;font-size:12px;margin-bottom:16px">Last updated: {ts}</p>\n' + build_financial_html(
    df_financial, tree
)

with open("financial_summary.html", "w", encoding="utf-8") as f:
    f.write(html)

print("Saved to financial_summary.html")

Saved to financial_summary.html
